# Query DSL — operator tour

This notebook walks through every operator in the query DSL
against the live corpus. Each section runs a small batch of
queries and tabulates the matching clip counts so you can see
what each operator does.

> New to the DevKit? Start with `01_quickstart.ipynb` for an
> introduction to the dataset and the DSL's place in it.

### Grammar in one screen

| form | meaning |
|---|---|
| `<entity>.<attr> <cmp> <value>` | basic attribute predicate |
| `<entity>(<expr>, …)` | same-entity grouping |
| `and`, `or`, `not` | boolean composition |
| `A while B` | A and B with intersecting intervals |
| `A then(K) B` | B starts during A or within K seconds after |
| `A because_of B` | A's `because_of` edge points at a B |
| `within W: E` | restrict E's time window to W's intervals |

The full specification lives in `docs/user/query_language.md`.


## Setup


In [1]:
import os
from pathlib import Path

from cascade_av.dataset import CascadeDataset

# --- shared notebook styling -----------------------------------------------
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
    "figure.dpi": 110,
    "savefig.dpi": 110,
})

pd.options.display.max_colwidth = 110
pd.options.display.width = 140
pd.options.display.float_format = "{:.2f}".format


In [2]:
ds = CascadeDataset(Path(os.environ["CASCADE_AV_DATASET_ROOT"]))

def run(queries: list[str]) -> pd.DataFrame:
    """Run a batch of DSL queries and tabulate clip counts."""
    return pd.DataFrame(
        [(q, ds.count(q)) for q in queries],
        columns=["query", "matching clips"],
    )


/home/horde/Projects/cascade-devkit-docs-demo-pose-attrs/src/cascade_av/dataset.py:118: UserWarning: 1 clip_id(s) had multiple annotation files; kept first by filename. (Set CASCADE_AV_VERBOSE=1 for per-file detail.)
  self._scan(annotation_paths, annotation_root)


## Attribute predicates

The atomic predicate is `entity.attribute <cmp> value`. The
comparison is `=` for equality and `<`, `<=`, `>`, `>=` for
ordered attributes.


In [3]:
run([
    "agent.type = ped",
    "ego.action = decel",
    "light.color = red",
    "env.lanes >= 2",
])


,query,matching clips
0,agent.type = ped,133
1,ego.action = decel,55
2,light.color = red,97
3,env.lanes >= 2,194


## Hierarchical aliases

Several attribute values are *parent aliases* that expand to a
union of children: `vehicle` (car, truck, bus, …),
`vru` (pedestrian, cyclist), `intersection` (4-way, T-junction,
roundabout, …), `turn` (left, right), `change_lane`
(change-left, change-right).


In [4]:
run([
    "agent.type = vehicle",
    "agent.type = vru",
    "agent.type = cyclist",
    "env.type = intersection",
])


,query,matching clips
0,agent.type = vehicle,313
1,agent.type = vru,144
2,agent.type = cyclist,34
3,env.type = intersection,210


## Set membership

`attr in (a, b, c)` is sugar for an OR over the listed values.


In [5]:
run([
    "ego.action in (stop, yield, decel)",
    "agent.type in (ped, cyclist, animal)",
])


,query,matching clips
0,"ego.action in (stop, yield, decel)",197
1,"agent.type in (ped, cyclist, animal)",144


## Same-entity coupling

Inside an `agent(…)` clause, every constraint applies to the
**same** agent. Free-floating predicates may bind to different
agents — note how the count changes.


In [6]:
run([
    "agent(type = vehicle, pos = front)",
    "agent.type = vehicle and agent.pos = front",
])


,query,matching clips
0,"agent(type = vehicle, pos = front)",303
1,agent.type = vehicle and agent.pos = front,307


## Ego-relative direction — `agent.dir`

Direction aliases: `same`, `opposite`, `perpendicular_lr`,
`perpendicular_rl`, and the parent `perpendicular` (union of
the two perpendicular leaves). Like `agent.pos`, `agent.dir`
samples at the agent's visibility-window midpoint.


In [7]:
run([
    "agent.dir = same",
    "agent.dir = opposite",
    "agent.dir = perpendicular",
    # Oncoming vehicle — the natural way to ask "is there an
    # oncoming car?" for unprotected-turn analysis.
    "agent(type = vehicle, dir = opposite)",
])


,query,matching clips
0,agent.dir = same,291
1,agent.dir = opposite,134
2,agent.dir = perpendicular,144
3,"agent(type = vehicle, dir = opposite)",107


## Per-interval pose — `pos_any` / `dir_any`

`agent.pos` / `agent.dir` sample at the **midpoint** of the
agent's visibility window (one deterministic value per
agent). For agents whose pose changes mid-window — e.g. a
vehicle that approaches in front and ends up perpendicular as
it crosses an intersection — the midpoint misses the
transient. `pos_any` / `dir_any` are `kind="list"` and match
if **any** `EgoRelativePose` interval carried the value.

The delta between the two flavors shows how often pose
changes during visibility:


In [8]:
# Midpoint vs interval — same query, different cardinality.
run([
    "agent.dir = perpendicular",      # midpoint
    "agent.dir_any = perpendicular",  # any interval
])


,query,matching clips
0,agent.dir = perpendicular,144
1,agent.dir_any = perpendicular,171


In [9]:
# Composite: vehicles that were either in front OR crossing
# at some point during their visibility.
run([
    "agent(type = vehicle, "
    "pos_any in (front, perpendicular_lr, perpendicular_rl))",
])


,query,matching clips
0,"agent(type = vehicle, pos_any in (front, perpendicular_lr, perpendicular_rl))",307


## Boolean operators

Standard `and`, `or`, `not` over any sub-expression.


In [10]:
run([
    "agent.type = ped and env.type = crosswalk",
    "ego.action = stop or ego.action = yield",
    "agent.type = ped and not env.type = crosswalk",
])


,query,matching clips
0,agent.type = ped and env.type = crosswalk,92
1,ego.action = stop or ego.action = yield,163
2,agent.type = ped and not env.type = crosswalk,41


## Action flags

Action types in the schema 2.0.0 corpus encode flags as
parenthesized suffixes on the `action_type` string itself
(`oxd:Walk (jaywalk)`). The legacy flag-field syntax
`action(jaywalk = true)` no longer parses — match the
suffix variants directly via literal-string equality or
`in (...)`.


In [11]:
run([
    "agent(type = ped, action.type in ("
    '"oxd:Walk (jaywalk)", "oxd:Walk (jaywalk, erratic)", '
    '"oxd:Run (jaywalk)", "oxd:Run (jaywalk, erratic)"))',
    "agent.action.type in ("
    '"oxd:Walk (erratic)", "oxd:Walk (jaywalk, erratic)", '
    '"oxd:Run (erratic)", "oxd:Run (jaywalk, erratic)")',
])


,query,matching clips
0,"agent(type = ped, action.type in (""oxd:Walk (jaywalk)"", ""oxd:Walk (jaywalk, erratic)"", ""oxd:Run (jaywalk)""...",35
1,"agent.action.type in (""oxd:Walk (erratic)"", ""oxd:Walk (jaywalk, erratic)"", ""oxd:Run (erratic)"", ""oxd:Run (...",1


## Temporal — `while`

`A while B` keeps only pairs of matches whose intervals
intersect.


In [12]:
run([
    "agent.type = ped while ego.action = decel",
    "light.color = red while ego.action = stop",
])


,query,matching clips
0,agent.type = ped while ego.action = decel,20
1,light.color = red while ego.action = stop,48


## Temporal — `then(K)`

`A then(K) B` keeps pairs where B starts during A or within
K seconds after A ends. Bare `then` (no parens) defaults to
K = 0 — touch or overlap only.


In [13]:
run([
    "light.color = yellow then(3) ego.action = stop",
    "light.color = green then ego.action = drive",
])


,query,matching clips
0,light.color = yellow then(3) ego.action = stop,6
1,light.color = green then ego.action = drive,76


## Relational — `because_of`

Follows the schema's `because_of` causal edge. Both halves are
emitted as match tuples so you can inspect cause and effect.


In [14]:
run([
    "ego.action = decel because_of agent.type = ped",
    "ego.action = drive because_of agent.type = ped",
])


,query,matching clips
0,ego.action = decel because_of agent.type = ped,0
1,ego.action = drive because_of agent.type = ped,0


## Scoping — `within W: E`

Restricts `E`'s evaluation window to `W`'s intervals. Inside,
`not E` means *E does not happen during W* — useful for
negative scenarios like "during red, ego never stops."


In [15]:
run([
    "within light.color = red: not ego.action = stop",
    "within env.type = crosswalk: agent.type = ped",
])


,query,matching clips
0,within light.color = red: not ego.action = stop,49
1,within env.type = crosswalk: agent.type = ped,87


## Annotator-tagged flags

Some attributes are not derived from kinematics but tagged by
the annotator (e.g. *"the ego could have safely cleared this
yellow"*). They compose with the same DSL.


In [16]:
run([
    "light(color = yellow, ego_in_on_yellow = true)",
    "light(color = yellow, could_have_cleared = true)",
])


,query,matching clips
0,"light(color = yellow, ego_in_on_yellow = true)",21
1,"light(color = yellow, could_have_cleared = true)",3


## Agent group size — `agent.amount`

Lone agent vs row/group vs traffic-density tier. Aliases:
`single`, `row` (alias `group`), `light_traffic`,
`medium_traffic`, `heavy_traffic`, plus the parents
`multiple` (everything but `single`) and `traffic` (the
three density tiers).


In [17]:
run([
    "agent.amount = single",
    "agent.amount = group",
    "agent(type = ped, amount = group)",
])


,query,matching clips
0,agent.amount = single,312
1,agent.amount = group,238
2,"agent(type = ped, amount = group)",68


## Signaling details on `agent.prop` / `ego.prop`

Properties of type `signal` carry a `signaling_details`
sub-object exposing the source modality (flashing light,
hand gesture, holding sign, other) and a `not_facing_ego`
boolean. Non-signal properties resolve to `None` so the
predicates fall through cleanly.


In [18]:
run([
    "agent.prop.source = flashing_light",
    "agent.prop.source in (flashing_light, holding_sign, other)",
    "agent.prop.not_facing_ego = false",
    "agent(prop.source = flashing_light)",
])


,query,matching clips
0,agent.prop.source = flashing_light,142
1,"agent.prop.source in (flashing_light, holding_sign, other)",156
2,agent.prop.not_facing_ego = false,158
3,agent(prop.source = flashing_light),142


## Containment flags — `lane_edge`, `near_lane`, `illegal_lane`

Boolean predicates that match existentially across the
entity's containment records. Exposed on both `agent` and
`obj`. `lane_edge` is true when any containment carries
`edge ∈ {"left","right"}`; `near_lane` and `illegal_lane`
mirror the boolean flags directly.


In [19]:
run([
    "agent.illegal_lane = true",
    "agent.lane_edge = true",
    "obj.lane_edge = true",
    "agent(type = car, illegal_lane = true)",
])


,query,matching clips
0,agent.illegal_lane = true,14
1,agent.lane_edge = true,58
2,obj.lane_edge = true,111
3,"agent(type = car, illegal_lane = true)",5


## Action ID-link lists — `action.link_to` / `action_target`

Two list-valued, ID-keyed attributes that complement the
`because_of` *operator*. `because_of` walks
`Action.because_of`; these expose the sibling ID lists for
explicit membership tests. UUIDs in the corpus carry hyphens
and must be quoted (the lexer treats bare `-` as an error).


In [20]:
# IDs are UUIDs, so the membership form is the natural call
# site.
run([
    'agent.action.link_to = "00000000-0000-0000-0000-000000000000"',
    'ego.action.action_target in ("00000000-0000-0000-0000-000000000000")',
])


,query,matching clips
0,"agent.action.link_to = ""00000000-0000-0000-0000-000000000000""",0
1,"ego.action.action_target in (""00000000-0000-0000-0000-000000000000"")",0


## Clip description — `clip.brief_description`

The annotator's one-line summary. Equality and IN-set only
for now; useful for spot-checks rather than aggregation
(each clip carries a unique string).


In [21]:
run([
    'clip.brief_description = "(no such description)"',
])


,query,matching clips
0,"clip.brief_description = ""(no such description)""",0


## Newly-aliased `"Other"` / `"Vehicle"` values

Four families that previously required the schema-literal
escape hatch (`agent.type = "Other"`) now resolve `other`
as a friendly alias. The annotator's generic-vehicle
fallback `"Vehicle"` (used when an agent is a vehicle that
doesn't match any specific subtype) is folded into the
`vehicle` parent alias and also reachable as the leaf
`generic_vehicle`.


In [22]:
run([
    "agent.type = other",
    "agent.type = generic_vehicle",
    "env.type = other",
    "agent.action.type = other",
])


,query,matching clips
0,agent.type = other,19
1,agent.type = generic_vehicle,0
2,env.type = other,27
3,agent.action.type = other,11


## Condition → environment lookup — `cond.env_*`

Each `Condition` carries an `env_id` pointing at one of the
bundle's `Environment` records. The DSL has no cross-entity
join operator today, so the four most-useful environment
fields are exposed as *denormalized* attributes on `cond`:
`cond.env_type` (reuses the `env_type` alias family),
`cond.env_lanes`, `cond.env_one_way`, and `cond.env_id`
(raw passthrough). A condition with an unresolvable
`env_id` falls through to `None` for the derived fields, so
predicates compose without an explicit guard.


In [23]:
run([
    "cond.env_type = road",
    "cond.type = construction and cond.env_type = road",
    "cond.env_one_way = true",
    "cond.env_lanes > 2",
])


,query,matching clips
0,cond.env_type = road,172
1,cond.type = construction and cond.env_type = road,62
2,cond.env_one_way = true,40
3,cond.env_lanes > 2,39
